In [0]:
%sql
CREATE TABLE IF NOT EXISTS homelab.gold.host_hourly (
  hour_ts TIMESTAMP, cpu_busy_avg DOUBLE, cpu_busy_p95 DOUBLE, mem_used_pct DOUBLE,
  load1_avg DOUBLE, temp_avg_c DOUBLE, temp_max_c DOUBLE, minutes_over_70c INT,
  net_rx_bytes_s DOUBLE, net_tx_bytes_s DOUBLE, samples INT);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS homelab.gold.container_daily (
  dt DATE, container STRING, cpu_cores_avg DOUBLE, cpu_cores_p95 DOUBLE,
  mem_mib_avg DOUBLE, mem_mib_max DOUBLE, samples INT);

In [0]:
%sql
CREATE OR REPLACE PROCEDURE homelab.gold.refresh_gold(IN p_from_date DATE)
LANGUAGE SQL SQL SECURITY INVOKER
AS BEGIN

  MERGE INTO homelab.gold.host_hourly t
  USING (
    SELECT date_trunc('hour', ts) AS hour_ts,
      avg(CASE WHEN metric_name='cpu_busy_ratio' THEN value END) AS cpu_busy_avg,
      percentile_approx(CASE WHEN metric_name='cpu_busy_ratio' THEN value END, 0.95) AS cpu_busy_p95,
      100 * (1 - avg(CASE WHEN metric_name='mem_available_bytes' THEN value END)
               / avg(CASE WHEN metric_name='mem_total_bytes' THEN value END)) AS mem_used_pct,
      avg(CASE WHEN metric_name='load1' THEN value END) AS load1_avg,
      avg(CASE WHEN metric_name='temp_celsius' THEN value END) AS temp_avg_c,
      max(CASE WHEN metric_name='temp_celsius' THEN value END) AS temp_max_c,
      CAST(sum(CASE WHEN metric_name='temp_celsius' AND value > 70 THEN 5 ELSE 0 END) AS INT) AS minutes_over_70c,
      sum(CASE WHEN metric_name='net_rx_bytes_per_s' THEN value END)
        / count(DISTINCT CASE WHEN metric_name='net_rx_bytes_per_s' THEN ts END) AS net_rx_bytes_s,
      sum(CASE WHEN metric_name='net_tx_bytes_per_s' THEN value END)
        / count(DISTINCT CASE WHEN metric_name='net_tx_bytes_per_s' THEN ts END) AS net_tx_bytes_s,
      CAST(count(DISTINCT ts) AS INT) AS samples
    FROM homelab.silver.prom_samples
    WHERE dt >= p_from_date AND metric_name IN
      ('cpu_busy_ratio','mem_available_bytes','mem_total_bytes','load1','temp_celsius',
       'net_rx_bytes_per_s','net_tx_bytes_per_s')
    GROUP BY date_trunc('hour', ts)
  ) s
  ON t.hour_ts = s.hour_ts
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *;

  MERGE INTO homelab.gold.container_daily t
  USING (
    SELECT dt, labels['name'] AS container,
      avg(CASE WHEN metric_name='container_cpu_cores' THEN value END) AS cpu_cores_avg,
      percentile_approx(CASE WHEN metric_name='container_cpu_cores' THEN value END, 0.95) AS cpu_cores_p95,
      avg(CASE WHEN metric_name='container_mem_bytes' THEN value END) / 1048576 AS mem_mib_avg,
      max(CASE WHEN metric_name='container_mem_bytes' THEN value END) / 1048576 AS mem_mib_max,
      CAST(count(DISTINCT ts) AS INT) AS samples
    FROM homelab.silver.prom_samples
    WHERE dt >= p_from_date AND labels['name'] IS NOT NULL
      AND metric_name IN ('container_cpu_cores','container_mem_bytes')
    GROUP BY dt, labels['name']
  ) s
  ON t.dt = s.dt AND t.container = s.container
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *;

  SELECT (SELECT count(*) FROM homelab.gold.host_hourly) AS host_hourly_rows,
         (SELECT count(*) FROM homelab.gold.container_daily) AS container_daily_rows;
END;

In [0]:
%sql
CALL homelab.gold.refresh_gold(DATE'2026-10-01');

In [0]:
%sql
CALL homelab.gold.refresh_gold(DATE'2026-10-01');

In [0]:
%sql
SELECT * FROM homelab.gold.host_hourly ORDER BY hour_ts DESC LIMIT 5;

In [0]:
%sql
SELECT * FROM homelab.gold.container_daily ORDER BY dt DESC, cpu_cores_avg DESC LIMIT 10;